In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLasso
import seaborn as sns
from datasets import *

In [2]:
def graphical_lasso_outage_detection(pre_outage_data, post_outage_data, alpha=1.0, noise_level=1e-5):
    """
    Apply Graphical LASSO to detect outage buses by comparing pre-outage and post-outage precision matrices.
    :param pre_outage_data: numpy array of shape (batch, channel, feature_len) representing pre-outage state
    :param post_outage_data: numpy array of shape (batch, channel, feature_len) representing post-outage state
    :param alpha: Regularization parameter for Graphical LASSO
    :return: Top two outage buses
    """
    batch, channel, feature_len = pre_outage_data.shape
    
    # Reshape to (T, d) where T=batch*channel and d=feature_len
    pre_data_reshaped = pre_outage_data.reshape(-1, feature_len)
    post_data_reshaped = post_outage_data.reshape(-1, feature_len)

    # Standardize the data
    scaler = StandardScaler()
    pre_data_reshaped = scaler.fit_transform(pre_data_reshaped)
    post_data_reshaped = scaler.transform(post_data_reshaped)
    
    # Estimate precision matrices using Graphical LASSO
    model_pre = GraphicalLasso(alpha=alpha).fit(pre_data_reshaped)
    model_post = GraphicalLasso(alpha=alpha).fit(post_data_reshaped)
    
    precision_pre = model_pre.precision_
    precision_post = model_post.precision_
    
    # Detect outage buses based on the difference in precision matrices
    diff_precision = np.abs(precision_post - precision_pre)
    top_outage_buses = np.argsort(np.sum(diff_precision, axis=0))[::-1] + 2
    
    return top_outage_buses

In [3]:
dataset = BinaryDataset('Node123_loop')

In [4]:
# Example usage with random data
pre_outage_data = dataset.data_df.values[:, None, :]  # Simulating (batch, channel, feature_len)
post_outage_data = dataset.data_outage_df.values[:, None, :]  # Simulating (batch, channel, feature_len)

top_outage_buses = graphical_lasso_outage_detection(pre_outage_data, post_outage_data)
print("Detected Outage Buses:", top_outage_buses)

Detected Outage Buses: [ 74  75  76  77 120 119 121 122 123 118 116 117 113 114 115 109 110 111
 112 108 103 104 105 106 107  73  72  78  79  80  81  82  66  83  67  92
  84  85  63  68  64  86  93  98  65  69  87  88  60  70  89  90  94  91
  99  71  61  95  62 100  96 101  97 102  59  58  57  16  17  20  18  19
  10  11  12   9  21  22  23  13  15  14  25  24   3   2   8  38  26  27
  28   4  29   5  33  39  34  35  37  30  36  40  31  41  32   6  42  43
  44  45   7  47  46  55  51  52  54  53  50  49  48  56]
